# StructureSAFE 配体和蛋白质编码与配体生成

本notebook将配体SDF文件和蛋白质PDB文件编码为向量，并使用StructureSAFE模型生成新的配体分子。

## 使用说明

1. **配置输入文件路径**：在下面的配置cell中设置输入文件路径和输出目录
2. **分步执行**：每个步骤可以在不同的conda环境中独立运行
3. **检查中间结果**：每个步骤完成后会保存中间结果，可以检查后再继续下一步

## 步骤概览

- **步骤1**：提取口袋PDB（基础环境：scipy, pandas, rdkit）
- **步骤2**：计算ligand_vec（RDKit + UniMol环境）
- **步骤3**：计算pocket_vec（Uni-Mol环境）
- **步骤4**：计算evo_vec（ESM-2环境）
- **步骤5**：计算IFP（ODDT环境）
- **步骤6**：生成配体（模型环境）



In [1]:
# ============================================================================
# 配置：设置输入文件路径和输出目录
# ============================================================================

import sys
from pathlib import Path

# 输入文件路径
LIGAND_SDF_PATH = "/home/yang2531/Documents/Project/Structure_safe/PAC1R/bay_structureSAFE.sdf"
PROTEIN_PDB_PATH = "/home/yang2531/Documents/Project/Structure_safe/PAC1R/PAC1R_StructureSAFE.pdb"

# 输出目录
OUTPUT_DIR = "/home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r"

# Uni-Mol相关文件路径（用于计算pocket_vec）
DICT_FILE = "/home/yang2531/Documents/Softwares/Uni-Mol/unimol/example_data/pocket/dict_coarse.txt"  # 请修改为实际路径
WEIGHTS = "/home/yang2531/Documents/Softwares/Uni-Mol/unimol/ckpt/pocket_pre_220816.pt"  # 请修改为实际路径

# 模型路径（用于生成配体）
MODEL_PATH = "/home/yang2531/Documents/Project/Structure_safe/outputs/models/12_19_25_300M_lr5e-5_KLnormclamp5_ufa_gate1_info0.10.2_xattn_new/checkpoint-33400/full_model"  # 请修改为实际路径

# 参数设置
POCKET_RADIUS = 10.0  # 口袋提取半径（Å）
NUM_SAMPLES = 10000  # 要生成的配体数量

# 创建输出目录
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"配置完成:")
print(f"  配体SDF: {LIGAND_SDF_PATH}")
print(f"  蛋白质PDB: {PROTEIN_PDB_PATH}")
print(f"  输出目录: {OUTPUT_DIR}")
print(f"  口袋半径: {POCKET_RADIUS}Å")
print(f"  生成数量: {NUM_SAMPLES}")



配置完成:
  配体SDF: /home/yang2531/Documents/Project/Structure_safe/PAC1R/bay_structureSAFE.sdf
  蛋白质PDB: /home/yang2531/Documents/Project/Structure_safe/PAC1R/PAC1R_StructureSAFE.pdb
  输出目录: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r
  口袋半径: 10.0Å
  生成数量: 10000


## 步骤1：提取口袋PDB

**环境要求**：基础环境（scipy, pandas, rdkit）

从SDF文件提取配体坐标，然后从PDB文件提取结合口袋（半径10Å内的残基），并保存为独立的PDB文件。

**输出**：
- `pocket.pdb`：提取的口袋PDB文件
- `receptor.pdb`：分离的受体PDB文件（仅包含ATOM记录）



In [4]:
# ============================================================================
# 步骤1：提取口袋PDB
# 环境：基础环境（scipy, pandas, rdkit）
# ============================================================================

import sys
from pathlib import Path

# 添加scripts目录到路径
sys.path.append(str(Path.cwd() / "scripts"))

from utils.encode_utils import (
    extract_ligand_coords_from_sdf,
    extract_pocket_from_pdb_with_ligand,
    separate_receptor_pdb
)

# 检查输入文件
ligand_sdf = Path(LIGAND_SDF_PATH)
protein_pdb = Path(PROTEIN_PDB_PATH)

if not ligand_sdf.exists():
    raise FileNotFoundError(f"配体SDF文件不存在: {ligand_sdf}")
if not protein_pdb.exists():
    raise FileNotFoundError(f"蛋白质PDB文件不存在: {protein_pdb}")

print("=" * 60)
print("步骤1：提取口袋PDB")
print("=" * 60)

# 1.1 从SDF提取配体坐标
print(f"\n1.1 从SDF提取配体坐标: {ligand_sdf}")
ligand_coords = extract_ligand_coords_from_sdf(str(ligand_sdf))
print(f"  提取到 {len(ligand_coords)} 个原子坐标")
print(f"  坐标范围: X[{ligand_coords[:, 0].min():.2f}, {ligand_coords[:, 0].max():.2f}], "
      f"Y[{ligand_coords[:, 1].min():.2f}, {ligand_coords[:, 1].max():.2f}], "
      f"Z[{ligand_coords[:, 2].min():.2f}, {ligand_coords[:, 2].max():.2f}]")

# 1.2 从PDB提取口袋
print(f"\n1.2 从PDB提取口袋 (半径={POCKET_RADIUS}Å): {protein_pdb}")
pocket_pdb_path = Path(OUTPUT_DIR) / "pocket.pdb"
extract_pocket_from_pdb_with_ligand(
    str(protein_pdb),
    ligand_coords,
    str(pocket_pdb_path),
    radius=POCKET_RADIUS,
    include_waters=False,
    exclude_h=False,
)
print(f"  ✓ 口袋PDB已保存: {pocket_pdb_path}")

# 1.3 分离受体PDB
print(f"\n1.3 分离受体PDB: {protein_pdb}")
receptor_pdb_path = Path(OUTPUT_DIR) / "receptor.pdb"
separate_receptor_pdb(
    str(protein_pdb),
    str(receptor_pdb_path),
)
print(f"  ✓ 受体PDB已保存: {receptor_pdb_path}")

print("\n" + "=" * 60)
print("步骤1完成！")
print("=" * 60)



步骤1：提取口袋PDB

1.1 从SDF提取配体坐标: /home/yang2531/Documents/Project/Structure_safe/PAC1R/bay_structureSAFE.sdf
  提取到 27 个原子坐标
  坐标范围: X[-11.85, 1.85], Y[-2.80, 3.39], Z[-27.31, -17.81]

1.2 从PDB提取口袋 (半径=10.0Å): /home/yang2531/Documents/Project/Structure_safe/PAC1R/PAC1R_StructureSAFE.pdb
  ✓ 口袋PDB已保存: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/pocket.pdb

1.3 分离受体PDB: /home/yang2531/Documents/Project/Structure_safe/PAC1R/PAC1R_StructureSAFE.pdb
  ✓ 受体PDB已保存: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/receptor.pdb

步骤1完成！


## 步骤2：计算ligand_vec

**环境要求**：RDKit + UniMol环境（rdkit, unimol_tools）

从SDF文件提取SMILES，使用UniMol计算配体embedding。

**输出**：
- `ligand_vec.npy`：配体向量（1536维）



In [2]:
# ============================================================================
# 步骤2：计算ligand_vec
# 环境：RDKit + UniMol环境（rdkit, unimol_tools）
# ============================================================================

import sys
from pathlib import Path
import numpy as np

# 添加utils目录到路径
# 获取项目根目录（notebook在scripts目录下，所以需要上一级目录）
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
utils_dir = project_root / "utils"
if utils_dir.exists():
    sys.path.insert(0, str(utils_dir))
else:
    # 如果不在scripts目录，尝试当前目录
    utils_dir = Path.cwd() / "utils"
    if utils_dir.exists():
        sys.path.insert(0, str(utils_dir))
    else:
        raise ImportError(f"Could not find utils directory. Tried: {project_root / 'utils'} and {Path.cwd() / 'utils'}")

from condition_extract import calc_smi_representation
from rdkit import Chem

# 检查输入文件
ligand_sdf = Path(LIGAND_SDF_PATH)
if not ligand_sdf.exists():
    raise FileNotFoundError(f"配体SDF文件不存在: {ligand_sdf}")

print("=" * 60)
print("步骤2：计算ligand_vec")
print("=" * 60)

# 2.1 从SDF提取SMILES
print(f"\n2.1 从SDF提取SMILES: {ligand_sdf}")
supplier = Chem.SDMolSupplier(str(ligand_sdf))
mol = None
for m in supplier:
    if m is not None:
        mol = m
        break

if mol is None:
    raise ValueError(f"无法从SDF文件读取有效分子: {ligand_sdf}")

# Remove hydrogens and convert to SMILES
mol = Chem.RemoveHs(mol)
smiles = Chem.MolToSmiles(mol)
print(f"  提取的SMILES: {smiles[:80]}...")

# 2.2 计算ligand_vec
print(f"\n2.2 使用UniMol计算配体embedding...")
ligand_vec_tensor = calc_smi_representation([smiles])
ligand_vec = ligand_vec_tensor.squeeze(0).numpy().astype(np.float32)

print(f"  ✓ ligand_vec计算完成")
print(f"    形状: {ligand_vec.shape}")
print(f"    均值: {ligand_vec.mean():.4f}")
print(f"    标准差: {ligand_vec.std():.4f}")

# 2.3 保存ligand_vec
ligand_vec_path = Path(OUTPUT_DIR) / "ligand_vec.npy"
np.save(str(ligand_vec_path), ligand_vec)
print(f"\n2.3 保存ligand_vec: {ligand_vec_path}")

print("\n" + "=" * 60)
print("步骤2完成！")
print("=" * 60)



步骤2：计算ligand_vec

2.1 从SDF提取SMILES: /home/yang2531/Documents/Project/Structure_safe/PAC1R/bay_structureSAFE.sdf
  提取的SMILES: NCC(=O)Nc1ccc(OCc2cc(C(F)(F)F)cc(C(F)(F)F)c2)cc1...

2.2 使用UniMol计算配体embedding...


2026-02-06 16:19:06 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /home/yang2531/anaconda3/envs/unimol_tool/lib/python3.11/site-packages/unimol_tools/weights/modelzoo/1.1B/checkpoint.pt
2026-02-06 16:19:13 | unimol_tools/data/conformer.py | 452 | INFO | Uni-Mol Tools | Start generating conformers...
1it [00:00,  1.44it/s]
2026-02-06 16:19:14 | unimol_tools/data/conformer.py | 467 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2026-02-06 16:19:14 | unimol_tools/data/conformer.py | 484 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 100.00% of molecules.
2026-02-06 16:19:14 | unimol_tools/tasks/trainer.py | 78 | INFO | Uni-Mol Tools | Number of GPUs available: 1
2026-02-06 16:19:14 | unimol_tools/tasks/trainer.py | 98 | INFO | Uni-Mol Tools | Using single GPU.
100%|██████████| 1/1 [00:00<00:00,  3.23it/s]

  ✓ ligand_vec计算完成
    形状: (1536,)
    均值: 0.0156
    标准差: 14.9522

2.3 保存ligand_vec: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/ligand_vec.npy

步骤2完成！


## 步骤3：计算pocket_vec

**环境要求**：Uni-Mol环境

使用Uni-Mol从口袋PDB计算binding pocket embedding。

**输入**：
- `pocket.pdb`：步骤1生成的口袋PDB文件
- `dict_file`：Uni-Mol字典文件路径
- `weights`：Uni-Mol模型权重文件路径

**输出**：
- `pocket_vec.npy`：口袋向量（512维）



In [2]:
# ============================================================================
# 步骤3：计算pocket_vec
# 环境：Uni-Mol环境
# ============================================================================

import sys
from pathlib import Path
import numpy as np

# 添加utils目录到路径
# 获取项目根目录（notebook在scripts目录下，所以需要上一级目录）
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
utils_dir = project_root / "utils"
if utils_dir.exists():
    sys.path.insert(0, str(utils_dir))
else:
    # 如果不在scripts目录，尝试当前目录
    utils_dir = Path.cwd() / "utils"
    if utils_dir.exists():
        sys.path.insert(0, str(utils_dir))
    else:
        raise ImportError(f"Could not find utils directory. Tried: {project_root / 'utils'} and {Path.cwd() / 'utils'}")

from build_pocket_lmdb_and_infer import write_lmdb_from_pdb, run_unimol_pocket_infer
from pocket_tensor import load_mol_repr_tensor

# 检查输入文件
pocket_pdb = Path(OUTPUT_DIR) / "pocket.pdb"
dict_file_path = Path(DICT_FILE)
weights_path = Path(WEIGHTS)

if not pocket_pdb.exists():
    raise FileNotFoundError(f"口袋PDB文件不存在: {pocket_pdb}")
if not dict_file_path.exists():
    raise FileNotFoundError(f"字典文件不存在: {dict_file_path}")
if not weights_path.exists():
    raise FileNotFoundError(f"权重文件不存在: {weights_path}")

print("=" * 60)
print("步骤3：计算pocket_vec")
print("=" * 60)

# 创建临时目录
temp_dir = Path(OUTPUT_DIR) / "temp_pocket"
temp_dir.mkdir(parents=True, exist_ok=True)
job_name = "pocket_compute"

# 3.1 创建LMDB
print(f"\n3.1 创建LMDB from pocket PDB...")
lmdb_path = write_lmdb_from_pdb(
    pdb_path=str(pocket_pdb),
    out_dir=str(temp_dir),
    job_name=job_name,
    radius=POCKET_RADIUS,
    include_waters=False,
    exclude_h=False
)
print(f"  ✓ LMDB已创建: {lmdb_path}")

# 3.2 运行Uni-Mol推理
print(f"\n3.2 运行Uni-Mol推理...")
print(f"  数据目录: {temp_dir}")
print(f"  字典文件: {dict_file_path}")
print(f"  权重文件: {weights_path}")
print(f"  结果目录: {temp_dir / 'results'}")

results_dir = str(temp_dir / "results")
try:
    pkl_path = run_unimol_pocket_infer(
        data_dir=str(temp_dir),
        job_name=job_name,
        dict_file=str(dict_file_path),
        weights=str(weights_path),
        results_dir=results_dir,
        batch_size=16,
        num_workers=4
    )
    print(f"  ✓ Uni-Mol推理完成: {pkl_path}")
except Exception as e:
    print(f"\n  ✗ Uni-Mol推理失败: {e}")
    print(f"\n  提示：如果Uni-Mol路径或环境有问题，您可以：")
    print(f"  1. 检查Uni-Mol目录是否存在: /home/yang2531/Documents/Softwares/Uni-Mol/unimol")
    print(f"  2. 检查字典文件和权重文件路径是否正确")
    print(f"  3. 确认当前conda环境已安装Uni-Mol相关依赖")
    print(f"  4. 查看上面的错误输出以获取更多信息")
    raise

# 3.3 加载pocket向量
print(f"\n3.3 加载pocket向量...")
pocket_tensor = load_mol_repr_tensor(pkl_path)
pocket_vec = pocket_tensor.numpy().astype(np.float32)

print(f"  ✓ pocket_vec计算完成")
print(f"    形状: {pocket_vec.shape}")
print(f"    均值: {pocket_vec.mean():.4f}")
print(f"    标准差: {pocket_vec.std():.4f}")

# 3.4 保存pocket_vec
pocket_vec_path = Path(OUTPUT_DIR) / "pocket_vec.npy"
np.save(str(pocket_vec_path), pocket_vec)
print(f"\n3.4 保存pocket_vec: {pocket_vec_path}")

print("\n" + "=" * 60)
print("步骤3完成！")
print("=" * 60)



ModuleNotFoundError: No module named 'lmdb'

## 步骤4：计算evo_vec

**环境要求**：ESM-2环境（esm, biopython）

使用ESM-2从完整蛋白质和口袋计算进化embedding。

**输入**：
- `receptor.pdb`：步骤1生成的受体PDB文件
- `pocket.pdb`：步骤1生成的口袋PDB文件

**输出**：
- `evo_vec.npy`：进化向量（1280维）



In [2]:
# ============================================================================
# 步骤4：计算evo_vec
# 环境：ESM-2环境（esm, biopython）
# ============================================================================

import sys
from pathlib import Path
import numpy as np
import torch

# 添加utils目录到路径
# 获取项目根目录（notebook在scripts目录下，所以需要上一级目录）
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
utils_dir = project_root / "utils"
if utils_dir.exists():
    sys.path.insert(0, str(utils_dir))
else:
    # 如果不在scripts目录，尝试当前目录
    utils_dir = Path.cwd() / "utils"
    if utils_dir.exists():
        sys.path.insert(0, str(utils_dir))
    else:
        raise ImportError(f"Could not find utils directory. Tried: {project_root / 'utils'} and {Path.cwd() / 'utils'}")

from esm_embedding_extract import ESM2PocketEmbedder

# 检查输入文件
receptor_pdb = Path(OUTPUT_DIR) / "receptor.pdb"
pocket_pdb = Path(OUTPUT_DIR) / "pocket.pdb"

if not receptor_pdb.exists():
    raise FileNotFoundError(f"受体PDB文件不存在: {receptor_pdb}")
if not pocket_pdb.exists():
    raise FileNotFoundError(f"口袋PDB文件不存在: {pocket_pdb}")

print("=" * 60)
print("步骤4：计算evo_vec")
print("=" * 60)

# 4.1 初始化ESM-2 embedder
print(f"\n4.1 初始化ESM-2 embedder...")
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = ESM2PocketEmbedder(
    model_name="esm2_t33_650M_UR50D",
    repr_layer=33,
    device=device
)
print(f"  ✓ ESM-2模型已加载 (device: {device})")

# 4.2 计算pocket embeddings
print(f"\n4.2 计算pocket embeddings...")
print(f"  完整蛋白质: {receptor_pdb}")
print(f"  口袋: {pocket_pdb}")
pocket_res_list, pocket_emb = embedder.pocket_embeddings(
    full_pdb_path=str(receptor_pdb),
    pocket_pdb_path=str(pocket_pdb),
    chain_id=None
)

# 4.3 计算均值embedding
print(f"\n4.3 计算均值embedding...")
evo_vec = pocket_emb.mean(dim=0).numpy().astype(np.float32)

print(f"  ✓ evo_vec计算完成")
print(f"    形状: {evo_vec.shape}")
print(f"    均值: {evo_vec.mean():.4f}")
print(f"    标准差: {evo_vec.std():.4f}")
print(f"    口袋残基数量: {len(pocket_res_list)}")

# 4.4 保存evo_vec
evo_vec_path = Path(OUTPUT_DIR) / "evo_vec.npy"
np.save(str(evo_vec_path), evo_vec)
print(f"\n4.4 保存evo_vec: {evo_vec_path}")

print("\n" + "=" * 60)
print("步骤4完成！")
print("=" * 60)



步骤4：计算evo_vec

4.1 初始化ESM-2 embedder...
  ✓ ESM-2模型已加载 (device: cuda)

4.2 计算pocket embeddings...
  完整蛋白质: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/receptor.pdb
  口袋: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/pocket.pdb

4.3 计算均值embedding...
  ✓ evo_vec计算完成
    形状: (1280,)
    均值: -0.0003
    标准差: 0.1375
    口袋残基数量: 87

4.4 保存evo_vec: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/evo_vec.npy

步骤4完成！


## 步骤5：计算IFP

**环境要求**：ODDT环境（oddt）

使用PLEC计算相互作用指纹（Interaction Fingerprint）。

**输入**：
- `receptor.pdb`：步骤1生成的受体PDB文件
- `ligand_sdf_path`：原始配体SDF文件路径

**输出**：
- `ifp.npy`：相互作用指纹（16384维）



## 步骤5前：Sanitize配体SDF文件

**环境要求**：RDKit环境（rdkit）

清理和sanitize配体SDF文件，修复可能的化学结构问题。

**输入**：
- 原始配体SDF文件

**输出**：
- Sanitized配体SDF文件（覆盖原文件或保存为新文件）



In [2]:
# 修复路径问题：在步骤6之前设置正确的项目路径
import sys
from pathlib import Path

# 获取项目根目录（notebook在scripts目录下，所以需要上一级目录）
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
src_dir = project_root / "src"
scripts_dir = project_root / "scripts"
utils_dir = project_root / "utils"

# 添加src、scripts和utils目录到路径
sys.path.insert(0, str(src_dir))
sys.path.insert(0, str(scripts_dir))
if utils_dir.exists():
    sys.path.insert(0, str(utils_dir))

print(f"项目根目录: {project_root}")
print(f"src目录: {src_dir}")
print(f"scripts目录: {scripts_dir}")
print(f"utils目录: {utils_dir}")
print(f"Python路径已更新")



项目根目录: /home/yang2531/Documents/Project/Structure_safe
src目录: /home/yang2531/Documents/Project/Structure_safe/src
scripts目录: /home/yang2531/Documents/Project/Structure_safe/scripts
utils目录: /home/yang2531/Documents/Project/Structure_safe/utils
Python路径已更新


In [3]:
# 临时修复：在步骤6之前，确保scripts_dir正确设置
# 如果之前的cell没有运行，这里会重新设置

import sys
from pathlib import Path

# 获取项目根目录
if Path.cwd().name == "scripts":
    project_root = Path.cwd().parent
else:
    project_root = Path.cwd()

scripts_dir = project_root / "scripts"
src_dir = project_root / "src"
utils_dir = project_root / "utils"

# 添加到全局变量
globals()['project_root'] = project_root
globals()['scripts_dir'] = scripts_dir
globals()['src_dir'] = src_dir
globals()['utils_dir'] = utils_dir

# 添加到sys.path
sys.path.insert(0, str(src_dir))
sys.path.insert(0, str(scripts_dir))
if utils_dir.exists():
    sys.path.insert(0, str(utils_dir))

print(f"✓ 路径已设置:")
print(f"  项目根目录: {project_root}")
print(f"  scripts_dir: {scripts_dir}")
print(f"  generate_from_pdb.py: {scripts_dir / 'generate_from_pdb.py'}")
print(f"  文件存在: {(scripts_dir / 'generate_from_pdb.py').exists()}")



✓ 路径已设置:
  项目根目录: /home/yang2531/Documents/Project/Structure_safe
  scripts_dir: /home/yang2531/Documents/Project/Structure_safe/scripts
  generate_from_pdb.py: /home/yang2531/Documents/Project/Structure_safe/scripts/generate_from_pdb.py
  文件存在: True


In [4]:
# ============================================================================
# 步骤5前：Sanitize配体SDF文件
# 环境：RDKit环境（rdkit）
# ============================================================================

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem

# 输入和输出文件路径
input_sdf = Path("/home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/bay_structureSAFE.sdf")
output_sdf = input_sdf.parent / f"{input_sdf.stem}_sanitized.sdf"

print("=" * 60)
print("Sanitize配体SDF文件")
print("=" * 60)

# 读取SDF文件
print(f"\n读取SDF文件: {input_sdf}")
supplier = Chem.SDMolSupplier(str(input_sdf))
mol = None
for m in supplier:
    if m is not None:
        mol = m
        break

if mol is None:
    raise ValueError(f"无法从SDF文件读取有效分子: {input_sdf}")

print(f"  原始分子原子数: {mol.GetNumAtoms()}")
print(f"  原始分子键数: {mol.GetNumBonds()}")

# Sanitize分子
print(f"\nSanitize分子...")
try:
    Chem.SanitizeMol(mol)
    print(f"  ✓ Sanitize成功")
except Exception as e:
    print(f"  ⚠️  Sanitize失败: {e}")
    print(f"  尝试移除氢原子后重新sanitize...")
    mol = Chem.RemoveHs(mol)
    try:
        Chem.SanitizeMol(mol)
        print(f"  ✓ 移除氢后sanitize成功")
    except Exception as e2:
        print(f"  ⚠️  移除氢后仍然失败: {e2}")
        print(f"  使用分子原样保存")

# 可选：生成3D坐标（如果需要）
print(f"\n生成3D坐标...")
try:
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    print(f"  ✓ 3D坐标生成成功")
except Exception as e:
    print(f"  ⚠️  3D坐标生成失败（使用原有坐标）: {e}")

# 保存sanitized分子
print(f"\n保存sanitized分子: {output_sdf}")
writer = Chem.SDWriter(str(output_sdf))
writer.write(mol)
writer.close()

print(f"  ✓ 保存完成")
print(f"\nSanitized文件: {output_sdf}")
print(f"  原子数: {mol.GetNumAtoms()}")
print(f"  键数: {mol.GetNumBonds()}")

# 显示SMILES
smiles = Chem.MolToSmiles(mol)
print(f"  SMILES: {smiles}")

print("\n" + "=" * 60)
print("Sanitize完成！")
print("=" * 60)
print(f"\n提示：如果要在步骤5中使用sanitized文件，请更新LIGAND_SDF_PATH为:")
print(f"  {output_sdf}")



Sanitize配体SDF文件

读取SDF文件: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/bay_structureSAFE.sdf
  原始分子原子数: 27
  原始分子键数: 28

Sanitize分子...
  ✓ Sanitize成功

生成3D坐标...
  ✓ 3D坐标生成成功

保存sanitized分子: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/bay_structureSAFE_sanitized.sdf
  ✓ 保存完成

Sanitized文件: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/bay_structureSAFE_sanitized.sdf
  原子数: 27
  键数: 28
  SMILES: NCC(=O)Nc1ccc(OCc2cc(C(F)(F)F)cc(C(F)(F)F)c2)cc1

Sanitize完成！

提示：如果要在步骤5中使用sanitized文件，请更新LIGAND_SDF_PATH为:
  /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/bay_structureSAFE_sanitized.sdf


In [3]:
# ============================================================================
# 步骤5：计算IFP
# 环境：ODDT环境（oddt）
# ============================================================================

import sys
from pathlib import Path
import numpy as np

# 添加utils目录到路径
# 获取项目根目录（notebook在scripts目录下，所以需要上一级目录）
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
utils_dir = project_root / "utils"
if utils_dir.exists():
    sys.path.insert(0, str(utils_dir))
else:
    # 如果不在scripts目录，尝试当前目录
    utils_dir = Path.cwd() / "utils"
    if utils_dir.exists():
        sys.path.insert(0, str(utils_dir))
    else:
        raise ImportError(f"Could not find utils directory. Tried: {project_root / 'utils'} and {Path.cwd() / 'utils'}")

from condition_extract import calc_ifp_plec

# 检查输入文件
receptor_pdb = Path(OUTPUT_DIR) / "receptor.pdb"
ligand_sdf = Path(LIGAND_SDF_PATH)

if not receptor_pdb.exists():
    raise FileNotFoundError(f"受体PDB文件不存在: {receptor_pdb}")
if not ligand_sdf.exists():
    raise FileNotFoundError(f"配体SDF文件不存在: {ligand_sdf}")

print("=" * 60)
print("步骤5：计算IFP")
print("=" * 60)

# 5.1 计算IFP
print(f"\n5.1 计算相互作用指纹 (PLEC)...")
print(f"  受体PDB: {receptor_pdb}")
print(f"  配体SDF: {ligand_sdf}")
ifp_tensor = calc_ifp_plec(str(receptor_pdb), str(ligand_sdf))

# 检查是否返回错误消息
if isinstance(ifp_tensor, str):
    raise ValueError(f"IFP计算失败: {ifp_tensor}")

ifp = ifp_tensor.numpy().astype(np.float32)

print(f"  ✓ IFP计算完成")
print(f"    形状: {ifp.shape}")
print(f"    均值: {ifp.mean():.4f}")
print(f"    标准差: {ifp.std():.4f}")
print(f"    非零元素: {(ifp > 0).sum()} / {len(ifp)}")

# 5.2 保存IFP
ifp_path = Path(OUTPUT_DIR) / "ifp.npy"
np.save(str(ifp_path), ifp)
print(f"\n5.2 保存IFP: {ifp_path}")

print("\n" + "=" * 60)
print("步骤5完成！")
print("=" * 60)



步骤5：计算IFP

5.1 计算相互作用指纹 (PLEC)...
  受体PDB: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/receptor.pdb
  配体SDF: /home/yang2531/Documents/Project/Structure_safe/PAC1R/bay_structureSAFE.sdf
  注意：如果遇到原子价错误，会自动sanitize配体后重试
  ✓ IFP计算完成
    形状: (16384,)
    均值: 0.0564
    标准差: 0.4364
    非零元素: 443 / 16384

5.2 保存IFP: /home/yang2531/Documents/Project/Structure_safe/outputs/encode_pac1r/ifp.npy

步骤5完成！


## 步骤6：生成配体

**环境要求**：模型环境（torch, transformers, 模型相关依赖）

加载所有计算得到的向量，使用StructureSAFE模型生成新的配体分子。

**输入**：
- `ligand_vec.npy`：步骤2生成的配体向量
- `pocket_vec.npy`：步骤3生成的口袋向量
- `evo_vec.npy`：步骤4生成的进化向量
- `ifp.npy`：步骤5生成的相互作用指纹
- `model_path`：训练好的模型路径

**输出**：
- `generated_molecules.txt`：生成的配体SMILES（每行一个）
- `generated_molecules.json`：生成的配体详细信息（JSON格式）



In [4]:
# ============================================================================
# 步骤6：生成配体
# 环境：模型环境（torch, transformers, 模型相关依赖）
# ============================================================================

import sys
import json
import logging
from pathlib import Path
import numpy as np
import torch

# 添加src和scripts目录到路径
sys.path.append(str(Path.cwd() / "src"))
sys.path.insert(0, str(Path.cwd() / "scripts"))

from data_loader.molecule_tokenizer import MoleculeTokenizer
from data_loader.utils import safe_to_smiles
from transformers import AutoTokenizer

# 导入generate_from_pdb中的函数
import importlib.util
generate_from_pdb_path = Path.cwd() / "scripts" / "generate_from_pdb.py"
spec = importlib.util.spec_from_file_location("generate_from_pdb", generate_from_pdb_path)
generate_from_pdb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generate_from_pdb)

detect_model_type = generate_from_pdb.detect_model_type
load_model_and_tokenizer = generate_from_pdb.load_model_and_tokenizer
generate_molecules_from_conditions = generate_from_pdb.generate_molecules_from_conditions

# 检查输入文件
ligand_vec_path = Path(OUTPUT_DIR) / "ligand_vec.npy"
pocket_vec_path = Path(OUTPUT_DIR) / "pocket_vec.npy"
evo_vec_path = Path(OUTPUT_DIR) / "evo_vec.npy"
ifp_path = Path(OUTPUT_DIR) / "ifp.npy"
model_path = Path(MODEL_PATH)

for vec_path in [ligand_vec_path, pocket_vec_path, evo_vec_path, ifp_path]:
    if not vec_path.exists():
        raise FileNotFoundError(f"向量文件不存在: {vec_path}")
if not model_path.exists():
    raise FileNotFoundError(f"模型路径不存在: {model_path}")

print("=" * 60)
print("步骤6：生成配体")
print("=" * 60)

# 设置日志
logger = logging.getLogger("generate_ligands")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# 6.1 加载所有向量
print(f"\n6.1 加载所有向量...")
ligand_vec = np.load(str(ligand_vec_path))
pocket_vec = np.load(str(pocket_vec_path))
evo_vec = np.load(str(evo_vec_path))
ifp = np.load(str(ifp_path))

print(f"  ✓ 所有向量已加载")
print(f"    ligand_vec: {ligand_vec.shape}")
print(f"    pocket_vec: {pocket_vec.shape}")
print(f"    evo_vec: {evo_vec.shape}")
print(f"    ifp: {ifp.shape}")

# 6.2 加载模型和tokenizer
print(f"\n6.2 加载模型和tokenizer: {model_path}")
model, tokenizer, device = load_model_and_tokenizer(str(model_path), logger)
print(f"  ✓ 模型已加载 (device: {device})")

# 6.3 生成配体
print(f"\n6.3 生成 {NUM_SAMPLES} 个配体分子...")
generated_molecules, conversion_stats = generate_molecules_from_conditions(
    model=model,
    tokenizer=tokenizer,
    pocket_vec=pocket_vec,
    evo_vec=evo_vec,
    ifp=ifp,
    ligand_vec=ligand_vec,
    num_samples=NUM_SAMPLES,
    device=device,
    logger=logger,
    batch_size=50,
    max_length=64,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
    max_retries=10,
    sample_posterior=False,
)

print(f"\n  ✓ 生成了 {len(generated_molecules)} 个有效配体分子")

# 6.4 保存结果
print(f"\n6.4 保存结果...")

# 保存为文本文件
txt_file = Path(OUTPUT_DIR) / "generated_molecules.txt"
with open(txt_file, "w", encoding="utf-8") as f:
    for mol in generated_molecules:
        f.write(f"{mol}\n")
print(f"  ✓ 保存到: {txt_file}")

# 保存为JSON文件
json_file = Path(OUTPUT_DIR) / "generated_molecules.json"
results = {
    "input_files": {
        "ligand_sdf": LIGAND_SDF_PATH,
        "protein_pdb": PROTEIN_PDB_PATH,
    },
    "model_path": str(model_path),
    "num_samples": len(generated_molecules),
    "generated_molecules": generated_molecules,
    "conversion_stats": conversion_stats,
    "condition_features": {
        "ligand_vec_shape": list(ligand_vec.shape),
        "pocket_vec_shape": list(pocket_vec.shape),
        "evo_vec_shape": list(evo_vec.shape),
        "ifp_shape": list(ifp.shape),
    },
    "generation_params": {
        "num_samples": NUM_SAMPLES,
        "batch_size": 50,
        "max_length": 64,
        "temperature": 1.0,
        "top_k": 50,
        "top_p": 0.95,
    }
}
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"  ✓ 保存到: {json_file}")

print("\n" + "=" * 60)
print("步骤6完成！")
print("=" * 60)
print(f"\n总共生成了 {len(generated_molecules)} 个有效配体分子")
print(f"结果保存在: {OUTPUT_DIR}")



/home/yang2531/anaconda3/envs/NovoMol/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: '/home/yang2531/Documents/Project/Structure_safe/scripts/scripts/generate_from_pdb.py'